In [ ]:
!pip install cvxpy

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

session = get_active_session()

long_df = session.table("FACTOR_MODEL_PROJECT.DBT_AURRIAGO.FACT_FACTOR_RETURNS").to_pandas()
wide_df = long_df.pivot(index="RETURN_DATE", columns="FACTOR_NAME", values="RETURN_VALUE").reset_index()
wide_df.columns.name = None
wide_df = wide_df.rename(columns={"RETURN_DATE": "Date"})
wide_df.columns = [c if c == "Date" else c.lower() for c in wide_df.columns]
wide_df["Date"] = pd.to_datetime(wide_df["Date"])
wide_df = wide_df[(wide_df["Date"] >= "1997-03-01") & (wide_df["Date"] <= "2012-12-01")].reset_index(drop=True)
# Restrict to March 1997-December 2012: the complete, gap-free period
# data source has missing values from ~2015 onward, which breaks scikit-learn (NaN not supported)

factorNames = ['world_equities', 'us_treasuries_10yr', 'high_yield',
               'inflation_protection', 'currency_protection']
assetNames = ['sp500_total_return', 'international_equity', 'us_treasury_20yr',
              'corporate_bond', 'commodity', 'tips']

In [ ]:
from factor_model_lib import run_factor_model, create_dictionary_for_analysis

factor_premia_annual = {
    'world_equities': 0.08, 'us_treasuries_10yr': 0.02, 'high_yield': 0.05,
    'inflation_protection': -0.003, 'currency_protection': 0.0
}
expected_returns_monthly = pd.DataFrame(
    [[factor_premia_annual[f] / 12 for f in factorNames]], columns=factorNames
)

def build_factor_matrix(data, assetNames, factorNames, method='CVLasso', methodOptions=None):
    rows = []
    for asset in assetNames:
        options = create_dictionary_for_analysis(method, methodOptions or {'n_folds': 5})
        options['date'] = 'Date'
        model = run_factor_model(data, asset, factorNames, method, options)
        implied_return_annual = 12 * model.predict(expected_returns_monthly)[0]
        rows.append([model.intercept_] + list(model.coef_) + [implied_return_annual])
    cols = ['Intercept'] + factorNames + ['Implied Expected Return (Annual)']
    return pd.DataFrame(rows, columns=cols, index=[a.replace('_', ' ').title() for a in assetNames]).round(3)

factor_matrix = build_factor_matrix(wide_df, assetNames, factorNames)
factor_matrix

In [ ]:
session.sql("USE DATABASE FACTOR_MODEL_PROJECT").collect()
session.sql("USE SCHEMA ANALYTICS").collect()

result_df = session.create_dataframe(factor_matrix.reset_index().rename(columns={'index': 'Asset'}))
result_df.write.save_as_table("FACTOR_MODEL_PROJECT.ANALYTICS.FACTOR_MATRIX_RESULTS", mode="overwrite")

In [ ]:
PALETTE = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']  # fixed factor order
loadings = factor_matrix[factorNames]

# Heatmap: assets x factors, diverging, centered at 0
fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(loadings, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            linewidths=2, linecolor="#fcfcfb", cbar_kws={'label': 'Loading'}, ax=ax)
ax.set_title("Factor Loadings — CV Lasso", color="#0b0b0b", fontsize=13)
ax.tick_params(colors="#52514e")
plt.tight_layout()
plt.show()

# Grouped bar chart: loadings per asset, one color per factor (fixed order)
fig, ax = plt.subplots(figsize=(10, 5))
loadings.plot(kind="bar", ax=ax, color=PALETTE, width=0.75)
ax.axhline(0, color="#c3c2b7", linewidth=1)
ax.set_title("Factor Loadings by Asset — CV Lasso", color="#0b0b0b", fontsize=13)
ax.set_ylabel("Loading")
ax.legend(title="Factor", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(axis="y", color="#e1e0d9", linewidth=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()